In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

print("=== FRONTIER_COVERAGE_AUDIT_WORLD2_KAGGLE_START ===")
print("python", sys.version)
subprocess.run(["nvidia-smi", "-L"], check=True)
import torch
print("cuda_available", torch.cuda.is_available())
print("cuda_device_count", torch.cuda.device_count())
assert torch.cuda.device_count() == 2
print(f"cuda_device_0={torch.cuda.get_device_name(0)}")
print(f"cuda_device_1={torch.cuda.get_device_name(1)}")
assert "T4" in torch.cuda.get_device_name(0)
assert "T4" in torch.cuda.get_device_name(1)

repo_url = "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"
repo_branch = "codex-architecture-v6-real-data-100-d300-b65536"
project_dir = pathlib.Path("/kaggle/working/CayleyBeam100H100_frontier_coverage_audit_w2")
if project_dir.exists():
    shutil.rmtree(project_dir)
print("GIT_CLONE_SOURCE", repo_url, repo_branch)
subprocess.run(["git", "clone", "--depth", "1", "--branch", repo_branch, repo_url, str(project_dir)], check=True)
print("PROJECT_DIR", project_dir)
required = ["setup.py", "production_v6_dispatcher.py", "data_loader.py", "beam_engine.py", "tests/frontier_coverage_audit_world2.py", "data/test.csv", "data/sample_submission.csv", "data/puzzle_info.json", "FullBeamNice/weights/p900-t000-q-sym_1777988767_best.pth", "third_party/cutlass/include/cutlass/gemm/device/gemm.h"]
for rel in required:
    print("required_file", rel, (project_dir / rel).exists())
    assert (project_dir / rel).exists()

os.chdir(project_dir)
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = "2"
os.environ["INFERENCE_BACKEND"] = "fullbeamnice_static"
os.environ["USE_CUDA_GRAPHS"] = "0"
os.environ["FULLBEAMNICE_DIR"] = str(project_dir / "FullBeamNice")
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_SOCKET_IFNAME"] = "lo"
os.environ["GLOO_SOCKET_IFNAME"] = "lo"
os.environ["FRONTIER_COVERAGE_TASK_COUNT"] = "10"
os.environ["FRONTIER_COVERAGE_MAX_DEPTH"] = "12"
os.environ["GLOBAL_BEAM_WIDTH"] = "65536"
os.environ["FRONTIER_COVERAGE_B_MICRO"] = "8192"

print("Step 1: build extension")
subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], check=True)
print("Step 2: frontier coverage audit WORLD2")
cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nnodes=1", "--nproc_per_node=2", "tests/frontier_coverage_audit_world2.py"]
completed = subprocess.run(cmd, check=False, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
print("returncode", completed.returncode)
assert completed.returncode == 0
assert completed.stdout.count("FRONTIER_COVERAGE_AUDIT_WORLD2_OK") == 2
assert "FRONTIER_COVERAGE_AUDIT_WORLD2_OK rank=0" in completed.stdout
assert "FRONTIER_COVERAGE_AUDIT_WORLD2_OK rank=1" in completed.stdout
assert "=== FRONTIER_COVERAGE_AUDIT_WORLD2_TEST_COMPLETE ===" in completed.stdout
output_path = pathlib.Path("/kaggle/working/frontier_coverage_audit_world2.csv")
audit_path = pathlib.Path("/kaggle/working/frontier_coverage_audit_world2.jsonl")
assert output_path.exists()
assert audit_path.exists()
row_count = sum(1 for _ in output_path.open("r", encoding="utf-8")) - 1
print("FRONTIER_COVERAGE_OUTPUT_ROWS", row_count)
assert row_count > 0
shutil.rmtree(project_dir, ignore_errors=True)
print("PROJECT_DIR_CLEANED", not project_dir.exists())
